In [1]:

import sys

import matplotlib
import numpy
import pandas
import sklearn

print(sys.version.split()[0], numpy.__version__, pandas.__version__, sklearn.__version__, matplotlib.__version__)


3.12.7 1.26.4 2.2.2 1.5.1 3.9.2


In [2]:

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 3039

data = pd.read_csv("data/spambase.csv")

feature_names = [c for c in data.columns if c != "spam"]
X, y = data[feature_names].to_numpy(), data["spam"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, stratify=y, random_state=SEED
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

print(X.shape, round(100 * y.mean(), 1), X_train.shape, X_test.shape)


(4204, 57) 39.9 (3153, 57) (1051, 57)


In [3]:

from sklearn.ensemble import AdaBoostClassifier, BaggingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.tree import DecisionTreeClassifier

tree = DecisionTreeClassifier(random_state=SEED)

bag = BaggingClassifier(
    DecisionTreeClassifier(),
    n_estimators=100,
    oob_score=True,
    random_state=SEED
)

rf = RandomForestClassifier(
    n_estimators=300,
    oob_score=True,
    random_state=SEED,
    n_jobs=-1
)

lr = make_pipeline(
    FunctionTransformer(np.log1p),
    StandardScaler(),
    LogisticRegression(max_iter=2000)
)

bag_lr = BaggingClassifier(lr, n_estimators=50, random_state=SEED)

knn = make_pipeline(
    FunctionTransformer(np.log1p),
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=15)
)

ada = AdaBoostClassifier(n_estimators=400, random_state=SEED)

members = [("lr", lr), ("knn", knn), ("tree", tree)]


SEED: 3039

# Part 1 — Bagging and Out-of-Bag Estimates
## 1.1 OOB Mask

In [5]:

def oob_mask(sample_indices, n_samples):
    """Return a boolean mask for rows not included in the bootstrap sample."""
    mask = np.ones(n_samples, dtype=bool)
    mask[np.asarray(sample_indices, dtype=int)] = False
    return mask


In [6]:

print(oob_mask([0, 2, 2], 5))
print(oob_mask([], 3))
print(oob_mask(np.array([3, 0]), 4))


[False  True False  True  True]
[ True  True  True]
[False  True  True False]


## 1.2 Out-of-Bag Share

In [7]:

bag.fit(X_train, y_train)

n = len(X_train)

oob_shares = []

for sample_indices in bag.estimators_samples_:
    mask = oob_mask(sample_indices, n)
    oob_shares.append(mask.mean())

mean_oob = np.mean(oob_shares)
mean_inbag = 1 - mean_oob

theoretical_oob = (1 - 1 / n) ** n
theoretical_inbag = 1 - theoretical_oob

print(f"Mean OOB share: {mean_oob:.4f}")
print(f"Mean in-bag share: {mean_inbag:.4f}")
print(f"Theoretical OOB share: {theoretical_oob:.4f}")
print(f"Theoretical in-bag share: {theoretical_inbag:.4f}")


Mean OOB share: 0.3680
Mean in-bag share: 0.6320
Theoretical OOB share: 0.3678
Theoretical in-bag share: 0.6322


Conclusion: The mean OOB share is 0.3680, which is very close to the theoretical value of 0.3678. The mean in-bag share is 0.6320, compared with the theoretical value of 0.6322. The results show that approximately 36.88% of training samples are not selected for each bootstrap sample. This agrees with the theoretical expectation. 

## 1.3 OOB Accuracy by Hand

In [8]:

votes = np.zeros((n, 2), dtype=int)

for tree_model, sample_indices in zip(
    bag.estimators_, bag.estimators_samples_
):
    mask = oob_mask(sample_indices, n)
    oob_indices = np.where(mask)[0]

    predictions = tree_model.predict(X_train[oob_indices])

    for row_idx, prediction in zip(oob_indices, predictions):
        votes[row_idx, int(prediction)] += 1

valid = votes.sum(axis=1) > 0

oob_predictions = np.argmax(votes[valid], axis=1)
manual_oob_accuracy = np.mean(oob_predictions == y_train[valid])

print(f"Manual OOB accuracy: {manual_oob_accuracy:.6f}")
print(f"Sklearn OOB accuracy: {bag.oob_score_:.6f}")
print(f"Rows with OOB votes: {valid.sum()} / {n}")


Manual OOB accuracy: 0.935617
Sklearn OOB accuracy: 0.935617
Rows with OOB votes: 3153 / 3153


Conclusion: The manually calculated OOB accuracy is 0.935617, which exactly matches th scikit-learn OOB score of 0.935617. All 3153 training samples received OOB votes. This confirms that our manual OOB voting procedure works correctly. 